# Basic Movements
In this file, i have calculated based on the following assumptions: 
1 vessel at quay with 50TEU load, 1 QC servicing and 10 prime movers supporting and 1 empty yard which the prime movers will be taking turns to fill up. 

For each TEU, create these activities:

qc_pick
qc_transfer
qc_load_to_pm
qc_travelback
pm_travel_to_yard
pm_unload
pm_return
And use dependencies:

qc_pick_n waits for previous qc_travelback_(n-1) if any
qc_transfer_n waits for qc_pick_n
qc_load_to_pm_n waits for:
qc_transfer_n
previous return of the assigned PM, if any
qc_travelback_n waits for qc_load_to_pm_n
pm_travel_to_yard_n waits for qc_load_to_pm_n
pm_unload_n waits for pm_travel_to_yard_n
pm_return_n waits for pm_unload_n
This way:

QC handoff is single, not duplicated
PM cycle starts from that handoff
QC can continue independently after travelback
PM can continue independently after return

In [15]:
import simpy
import shapely.geometry
import pandas as pd
import re
import openclsim.core as core
import openclsim.model as model

# --------------------------------------------------
# 1. Initialise simpy environment and registry
# --------------------------------------------------
my_env = simpy.Environment(initial_time=0)
registry = {}

# --------------------------------------------------
# 2. Geometry
# --------------------------------------------------
quay_loc = shapely.geometry.Point(4.18055556, 52.18664444)
vessel_loc = shapely.geometry.Point(4.18055556, 52.18664444)
yard_loc = shapely.geometry.Point(4.25222222, 52.11428333)

# --------------------------------------------------
# 3. Define object classes
# --------------------------------------------------
Site = type(
    "Site",
    (
        core.Identifiable,
        core.Log,
        core.Locatable,
        core.HasContainer,
        core.HasResource,
    ),
    {},
)

VesselUnloader = type(
    "VesselUnloader",
    (
        core.Identifiable,
        core.ContainerDependentMovable,
        core.HasResource,
        core.Processor,
        core.LoadingFunction,
        core.UnloadingFunction,
    ),
    {},
)

LandTransporter = type(
    "LandTransporter",
    (
        core.Identifiable,
        core.ContainerDependentMovable,
        core.Processor,
        core.HasResource,
        core.LoadingFunction,
        core.UnloadingFunction,
    ),
    {},
)

# --------------------------------------------------
# 4. Create fixed objects
# --------------------------------------------------
quay_site = Site(
    env=my_env,
    name="Quay Site",
    geometry=quay_loc,
    capacity=1000,
    level=0,
)

yard_block = Site(
    env=my_env,
    name="YardBlockA",
    geometry=yard_loc,
    capacity=1000,
    level=0,
)

initial_teu = 50

vessel01 = VesselUnloader(
    env=my_env,
    name="vessel01",
    geometry=vessel_loc,
    loading_rate=1,
    unloading_rate=1,
    capacity=initial_teu,
    level=initial_teu,
    compute_v=lambda x: 10,
)

qc01 = VesselUnloader(
    env=my_env,
    name="qc_01",
    geometry=quay_loc,
    loading_rate=1,
    unloading_rate=1,
    capacity=1,
    level=0,
    compute_v=lambda x: 10,
)

# --------------------------------------------------
# 5. Parameters
# --------------------------------------------------
fleet_size = 10

# QC timings
qc_pick_time = 60
qc_transfer_time = 120
qc_load_to_pm_time = 60
qc_travelback_time = 60

# PM timings
pm_unload_time = 120
pm_speed = 5

# --------------------------------------------------
# 6. Create PM fleet
# --------------------------------------------------
pm_fleet = []
for i in range(1, fleet_size + 1):
    pm = LandTransporter(
        env=my_env,
        name=f"pm_{i:02d}",
        geometry=quay_loc,
        loading_rate=1,
        unloading_rate=1,
        capacity=1,
        level=0,
        compute_v=lambda x, v=pm_speed: v,
    )
    pm_fleet.append(pm)

# --------------------------------------------------
# 7. Create activities
# --------------------------------------------------
qc_pick_activities = []
qc_transfer_activities = []
qc_load_to_pm_activities = []
qc_travelback_activities = []
pm_travel_activities = []
pm_unload_activities = []
pm_return_activities = []
all_jobs = []

last_qc_travelback_name = None
last_pm_return_name = {pm.name: None for pm in pm_fleet}

for n in range(1, initial_teu + 1):
    mover = pm_fleet[(n - 1) % fleet_size]

    qc_pick_start_events = []
    if last_qc_travelback_name is not None:
        qc_pick_start_events.append(
            {"name": last_qc_travelback_name, "type": "activity", "state": "done"}
        )

    qc_pick = model.ShiftAmountActivity(
        env=my_env,
        name=f"TEU_{n}_qc_pick_{mover.name}",
        registry=registry,
        processor=qc01,
        origin=vessel01,
        destination=qc01,
        amount=1,
        duration=qc_pick_time,
        start_event=qc_pick_start_events,
    )

    qc_transfer = model.BasicActivity(
        env=my_env,
        name=f"TEU_{n}_qc_transfer_{mover.name}",
        registry=registry,
        duration=qc_transfer_time,
        start_event=[
            {"name": f"TEU_{n}_qc_pick_{mover.name}", "type": "activity", "state": "done"}
        ],
    )

    qc_load_start_events = [
        {"name": f"TEU_{n}_qc_transfer_{mover.name}", "type": "activity", "state": "done"}
    ]

    if last_pm_return_name[mover.name] is not None:
        qc_load_start_events.append(
            {"name": last_pm_return_name[mover.name], "type": "activity", "state": "done"}
        )

    qc_load_to_pm = model.ShiftAmountActivity(
        env=my_env,
        name=f"TEU_{n}_qc_load_to_pm_{mover.name}",
        registry=registry,
        processor=qc01,
        origin=qc01,
        destination=mover,
        amount=1,
        duration=qc_load_to_pm_time,
        start_event=qc_load_start_events,
    )

    qc_travelback = model.BasicActivity(
        env=my_env,
        name=f"TEU_{n}_qc_travelback_{mover.name}",
        registry=registry,
        duration=qc_travelback_time,
        start_event=[
            {"name": f"TEU_{n}_qc_load_to_pm_{mover.name}", "type": "activity", "state": "done"}
        ],
    )

    pm_travel_to_yard = model.MoveActivity(
        env=my_env,
        name=f"TEU_{n}_pm_travel_to_yard_{mover.name}",
        registry=registry,
        mover=mover,
        destination=yard_block,
        start_event=[
            {"name": f"TEU_{n}_qc_load_to_pm_{mover.name}", "type": "activity", "state": "done"}
        ],
    )

    pm_unload = model.ShiftAmountActivity(
        env=my_env,
        name=f"TEU_{n}_pm_unload_{mover.name}",
        registry=registry,
        processor=mover,
        origin=mover,
        destination=yard_block,
        amount=1,
        duration=pm_unload_time,
        start_event=[
            {"name": f"TEU_{n}_pm_travel_to_yard_{mover.name}", "type": "activity", "state": "done"}
        ],
    )

    pm_return = model.MoveActivity(
        env=my_env,
        name=f"TEU_{n}_pm_return_{mover.name}",
        registry=registry,
        mover=mover,
        destination=quay_site,
        start_event=[
            {"name": f"TEU_{n}_pm_unload_{mover.name}", "type": "activity", "state": "done"}
        ],
    )

    job = model.ParallelActivity(
        env=my_env,
        name=f"TEU_{n}_job_{mover.name}",
        registry=registry,
        sub_processes=[
            qc_pick,
            qc_transfer,
            qc_load_to_pm,
            qc_travelback,
            pm_travel_to_yard,
            pm_unload,
            pm_return,
        ],
    )

    all_jobs.append(job)

    qc_pick_activities.append(qc_pick)
    qc_transfer_activities.append(qc_transfer)
    qc_load_to_pm_activities.append(qc_load_to_pm)
    qc_travelback_activities.append(qc_travelback)
    pm_travel_activities.append(pm_travel_to_yard)
    pm_unload_activities.append(pm_unload)
    pm_return_activities.append(pm_return)

    last_qc_travelback_name = f"TEU_{n}_qc_travelback_{mover.name}"
    last_pm_return_name[mover.name] = f"TEU_{n}_pm_return_{mover.name}"

# --------------------------------------------------
# 8. Combine system
# --------------------------------------------------
system = model.ParallelActivity(
    env=my_env,
    name="terminal_fleet_system",
    registry=registry,
    sub_processes=all_jobs,
)

# --------------------------------------------------
# 9. Register and run
# --------------------------------------------------
model.register_processes([system])

for pm in pm_fleet:
    print(f"{pm.name} initial location: {pm.geometry}")

my_env.run()

# --------------------------------------------------
# 10. Inspect end results
# --------------------------------------------------
print("Vessel:", vessel01.container.get_level())
print("QC:", qc01.container.get_level())
print("Quay:", quay_site.container.get_level())
print("Yard:", yard_block.container.get_level())

for pm in pm_fleet:
    print(f"{pm.name} final location: {pm.geometry}")

# --------------------------------------------------
# 11. Build detailed ordered activity event log
# --------------------------------------------------
def parse_activity_metadata(activity_name):
    patterns = [
        (r"TEU_(\d+)_qc_pick_(pm_\d+)", "qc_pick", 1),
        (r"TEU_(\d+)_qc_transfer_(pm_\d+)", "qc_transfer", 2),
        (r"TEU_(\d+)_qc_load_to_pm_(pm_\d+)", "qc_load_to_pm", 3),
        (r"TEU_(\d+)_qc_travelback_(pm_\d+)", "qc_travelback", 4),
        (r"TEU_(\d+)_pm_travel_to_yard_(pm_\d+)", "pm_travel_to_yard", 5),
        (r"TEU_(\d+)_pm_unload_(pm_\d+)", "pm_unload", 6),
        (r"TEU_(\d+)_pm_return_(pm_\d+)", "pm_return", 7),
    ]

    for pattern, activity_type, activity_order in patterns:
        m = re.search(pattern, activity_name)
        if m:
            return int(m.group(1)), m.group(2), activity_type, activity_order

    return None, None, "unknown", 999

activity_frames = []
all_activities = (
    qc_pick_activities
    + qc_transfer_activities
    + qc_load_to_pm_activities
    + qc_travelback_activities
    + pm_travel_activities
    + pm_unload_activities
    + pm_return_activities
)

for act in all_activities:
    if hasattr(act, "log"):
        df = pd.DataFrame(act.log)
        if not df.empty:
            teu_id, pm_name, activity_type, activity_order = parse_activity_metadata(act.name)
            df = df.copy()
            df["activity_name"] = act.name
            df["teu_id"] = teu_id
            df["pm_name"] = pm_name
            df["activity_type"] = activity_type
            df["activity_order"] = activity_order
            activity_frames.append(df)

if activity_frames:
    activity_logs_df = pd.concat(activity_frames, ignore_index=True)
else:
    activity_logs_df = pd.DataFrame()

activity_logs_df = activity_logs_df.drop(
    columns=[c for c in ["ActivityID", "ObjectState", "ActivityLabel"] if c in activity_logs_df.columns],
    errors="ignore"
)

if not activity_logs_df.empty:
    activity_logs_df = activity_logs_df[
        activity_logs_df["ActivityState"].isin(["WAIT_START", "WAIT_STOP", "START", "STOP"])
    ].copy()

    activity_logs_df = activity_logs_df[activity_logs_df["teu_id"].notna()].copy()

    activity_logs_df = activity_logs_df.sort_values(
        by=["teu_id", "activity_order", "Timestamp"]
    ).reset_index(drop=True)

    print(activity_logs_df.head(50))
    activity_logs_df.to_csv("activity_sequence_logs.csv", index=False)
else:
    print("No activity logs found.")

# --------------------------------------------------
# 12. Build compact one-row-per-TEU summary
# --------------------------------------------------
activity_summary_milestones_df = activity_summary_milestones_df.round(2)

def minutes_to_mmss(x):
    if pd.isna(x):
        return None
    total_seconds = int(round(x * 60))
    minutes = total_seconds // 60
    seconds = total_seconds % 60
    return f"{minutes:02d}:{seconds:02d}"

formatted_df = activity_summary_milestones_df.copy()

time_cols = [
    "qc_pick_start_min",
    "qc_load_to_pm_stop_min",
    "pm_travel_to_yard_stop_min",
    "pm_unload_stop_min",
    "pm_return_stop_min",
]

for col in time_cols:
    formatted_df[col] = formatted_df[col].apply(minutes_to_mmss)

print(formatted_df.to_string(index=False))
formatted_df.to_csv("activity_summary_milestones_formatted.csv", index=False)

pm_01 initial location: POINT (4.18055556 52.18664444)
pm_02 initial location: POINT (4.18055556 52.18664444)
pm_03 initial location: POINT (4.18055556 52.18664444)
pm_04 initial location: POINT (4.18055556 52.18664444)
pm_05 initial location: POINT (4.18055556 52.18664444)
pm_06 initial location: POINT (4.18055556 52.18664444)
pm_07 initial location: POINT (4.18055556 52.18664444)
pm_08 initial location: POINT (4.18055556 52.18664444)
pm_09 initial location: POINT (4.18055556 52.18664444)
pm_10 initial location: POINT (4.18055556 52.18664444)
Vessel: 0
QC: 0
Quay: 0
Yard: 50
pm_01 final location: POINT (4.18055556 52.18664444)
pm_02 final location: POINT (4.18055556 52.18664444)
pm_03 final location: POINT (4.18055556 52.18664444)
pm_04 final location: POINT (4.18055556 52.18664444)
pm_05 final location: POINT (4.18055556 52.18664444)
pm_06 final location: POINT (4.18055556 52.18664444)
pm_07 final location: POINT (4.18055556 52.18664444)
pm_08 final location: POINT (4.18055556 52.186

PermissionError: [Errno 13] Permission denied: 'activity_summary_milestones_formatted.csv'